In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement

## Definir la source en fonction de l'environnement
TODO: adapter `source_catalog` si ce nouveau projet pointe vers un catalog different de l'ancien.

In [0]:
if current_environment == 'preprd':
    source_catalog = "ext_mal_psql_maite_vision_board_test.public"
else:
    source_catalog = f"ext_mal_psql_maite_vision_board_{current_environment}.public"

## Chargement des tables sources
Pur chargement, aucune transformation ici (coherent avec le pattern de l'ancien projet).

In [0]:
batches = spark.table(f"{source_catalog}.batches")
plants_production_lines = spark.table(f"{source_catalog}.plants_production_lines")
parameters_variables = spark.table(f"{source_catalog}.parameters_variables")
energy_metadata = spark.table(f"mal_maite_common_{current_environment}.gold.energy_metadata")
batches_production_monitoring = spark.table(f"{source_catalog}.batches_production_monitoring")
manual_entries = spark.table(f"{source_catalog}.manual_entries")
parameters_localizations = spark.table(f"{source_catalog}.parameters_localizations")
requirement_specifications = spark.table(f"{source_catalog}.requirement_specifications")
goods_varieties = spark.table(f"{source_catalog}.goods_varieties")

# localization_events existe en 1 table par site (comme les mesures dans l'ancien projet)
sites = ["rouen1", "nogent1", "nogent2", "prouvy1", "strasbourg1", "strasbourg2", "polisy1", "buzau1", "bolelemi1"]

# IMPORTANT : select explicite par nom de colonne, pas juste withColumn/table brute.
# Les tables par site sont gerees independamment et peuvent avoir un ordre de
# colonnes different d'un site a l'autre. Comme l'union plus loin (UNION ALL en
# SQL) fonctionne par POSITION et non par nom, un ordre different provoque un
# decalage silencieux des valeurs entre colonnes (ex: batch_id qui se retrouve
# nul car alignee avec une autre colonne vide chez un site donne).
localization_events_by_site = {
    site: spark.table(f"mal_maite_{site}_{current_environment}.gold.localization_events")
          .select(
              "prd_line",
              "prd_workshop",
              "prd_cell",
              "batch_id",
              "localization_event",
              "start",
              "end"
          )
          .withColumn("site", F.lit(site))
    for site in sites
}